<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/baseline/notebooks/05_ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Ensemble and Submission
Stacked BiLSTM + Stacked BiGRU


## Environment Setup

In [1]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)

/content/UA-Locations-NER-Deep-Learning


In [2]:
%pip install -q -r "$repo_dir/requirements.txt"

In [ ]:
import json

import pandas as pd
from google.colab import drive

from src.data_io import get_data_dir
from src.metrics import comp_metric, hard_processing, location_errors
from src.postprocessing import postprocess

## Load data

In [ ]:
drive.mount("/content/drive")

RESULTS_DIR = Path("/content/drive/MyDrive/ner_results")
MODEL_DIRS = {"bilstm": RESULTS_DIR / "bilstm_2l_h256_s17_e4", "bigru": RESULTS_DIR / "bigru_2l_h256_s17_e3"}

def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]

VAL_GOLD_PATH = RESULTS_DIR / "shared/val_gold.jsonl"

val_gold = load_jsonl(VAL_GOLD_PATH)
val_predictions = {name: load_jsonl(folder / "val.jsonl") for name, folder in MODEL_DIRS.items()}
test_predictions = {name: load_jsonl(folder / "test.jsonl") for name, folder in MODEL_DIRS.items()}

Fixing rows order:

In [ ]:
gold_ids = [row["text_id"] for row in val_gold]

for name, rows in val_predictions.items():
    by_id = {row["text_id"]: row for row in rows}

    if len(by_id) != len(rows):
        raise ValueError(f"{name}: duplicate ID")
    if set(by_id) != set(gold_ids):
        raise ValueError(f"{name}: lost or extra ID")

    rows = [by_id[text_id] for text_id in gold_ids]

    for gold, prediction in zip(val_gold, rows):
        if gold["text"] != prediction["text"]:
            raise ValueError(f"{name}: texts do not match")

    val_predictions[name] = rows

Validation spans:

In [ ]:
for rows in [val_gold, *val_predictions.values()]:
    for row in rows:
        for start, end in row["spans"]:
            if not 0 <= start < end <= len(row["text"]):
                raise ValueError(f"Incorrect span: {row['text_id']}")

## Baseline metrics

In [ ]:
CLEAN_OPTIONS = {"expand": False, "trim": True, "empty": True}

y_true = []
for row in val_gold:
    y_true.append([row["text"][a:b] for a, b in row["spans"]])

val_locations = {}
for name, rows in val_predictions.items():
    predictions = []
    for row in rows:
        locations = postprocess(row["text"], row["spans"], **CLEAN_OPTIONS)
        predictions.append(locations)
    val_locations[name] = predictions

baseline_scores = {}
for name, predictions in val_locations.items():
    baseline_scores[name] = comp_metric(y_true, predictions)

pd.Series(baseline_scores, name="validation_f1").to_frame()

## Diversity & errors

In [ ]:
def normalize_set(locations):
    return {hard_processing(location) for location in locations}

gold_sets = list(map(normalize_set, y_true))
model_sets = {}
for name, predictions in val_locations.items():
    model_sets[name] = list(map(normalize_set, predictions))

**Findings:**

In [ ]:
comparison = {
    "both_right": 0,
    "only_bilstm_right": 0,
    "only_bigru_right": 0,
    "both_wrong": 0,
}

for gold, lstm, gru in zip(gold_sets, model_sets["bilstm"], model_sets["bigru"]):
    if lstm == gold and gru == gold:
        comparison["both_right"] += 1
    elif lstm == gold:
        comparison["only_bilstm_right"] += 1
    elif gru == gold:
        comparison["only_bigru_right"] += 1
    else:
        comparison["both_wrong"] += 1

pd.Series(comparison, name="texts").to_frame()

In [ ]:
examples = []

for i, row in enumerate(val_gold):
    lstm = val_locations["bilstm"][i]
    gru = val_locations["bigru"][i]

    if model_sets["bilstm"][i] == gold_sets[i] and model_sets["bigru"][i] == gold_sets[i]:
        continue

    examples.append({
        "text_id": row["text_id"],
        "text": row["text"],
        "gold": y_true[i],
        "bilstm": lstm,
        "bilstm_errors": location_errors(y_true[i], lstm),
        "bigru": gru,
        "bigru_errors": location_errors(y_true[i], gru)})

    if len(examples) == 5:
        break

display(pd.DataFrame(examples))

Error correlation:

In [ ]:
errors = {}

for name, predictions in model_sets.items():
    errors[name] = []
    for gold, predicted in zip(gold_sets, predictions):
        errors[name].append(int(predicted != gold))

errors = pd.DataFrame(errors)

display(errors.mean().rename("text_error_rate").to_frame())
display(errors.corr())

**Findings:**

## Ensembling
Union vs. Intersection

In [ ]:
def combine_predictions(predictions, method):
    result = []

    for rows in zip(*predictions.values()):
        location_sets = list(map(normalize_set, rows))
        if method == "union":
            selected = set.union(*location_sets)
        elif method == "intersection":
            selected = set.intersection(*location_sets)
        else:
            raise ValueError(method)

        originals = {}
        for locations in rows:
            for location in locations:
                originals.setdefault(hard_processing(location), location)
        result.append([originals[key] for key in sorted(selected)])

    return result


val_ensembles = {
    "union": combine_predictions(val_locations, "union"),
    "intersection": combine_predictions(val_locations, "intersection")
}

In [ ]:
scores = baseline_scores.copy()
for name, predictions in val_ensembles.items():
    scores[name] = comp_metric(y_true, predictions)

pd.Series(scores, name="validation_f1").to_frame()

**Findings:**

## Weight search

BiLSTM and BiGRU weights: w and 1 − w. Voting threshold — ≥ 0.5. We compare the weights only on the VAL.


In [ ]:
def weighted_predictions(predictions, weights):
    result = []

    for rows in zip(*predictions.values()):
        votes = {}
        originals = {}

        for name, locations in zip(predictions, rows):
            for key in normalize_set(locations):
                votes[key] = votes.get(key, 0) + weights[name]

            for location in locations:
                originals.setdefault(hard_processing(location), location)

        selected = []
        for key, vote in sorted(votes.items()):
            if vote >= 0.5:
                selected.append(originals[key])

        result.append(selected)
    return result

In [ ]:
WEIGHT_GRID = [0, 0.25, 0.5, 0.75, 1]

weight_results = []
for w in WEIGHT_GRID:
    weights = {"bilstm": w, "bigru": 1 - w}
    predictions = weighted_predictions(val_locations, weights)

    weight_results.append({
        "bilstm_weight": w,
        "bigru_weight": 1 - w,
        "validation_f1": comp_metric(y_true, predictions),
    })

pd.DataFrame(weight_results)

## Comparison

In [ ]:
reference_name = max(baseline_scores, key=baseline_scores.get)
reference = val_locations[reference_name]

comparison = []

for name, predictions in val_ensembles.items():
    fixed = new = 0

    for gold, before, after in zip(y_true, reference, predictions):
        before_errors = location_errors(gold, before)
        after_errors = location_errors(gold, after)

        for kind in ["false_positives", "false_negatives"]:
            old = set(before_errors[kind])
            current = set(after_errors[kind])

            fixed += len(old - current)
            new += len(current - old)

    comparison.append({
        "ensemble": name,
        "validation_f1": scores[name],
        "fixed_errors": fixed,
        "new_errors": new,
    })

print("Reference model:", reference_name)
pd.DataFrame(comparison)

Findings:

## Submission

In [ ]:
data_dir = get_data_dir(ENVIRONMENT)
test = pd.read_csv(data_dir / "test.csv")
test_ids = test["text_id"].tolist()

for name, rows in test_predictions.items():
    by_id = {row["text_id"]: row for row in rows}

    if len(by_id) != len(rows):
        raise ValueError(f"{name}: duplicate test ID")
    if set(by_id) != set(test_ids):
        raise ValueError(f"{name}: missing or extra test ID")

    rows = [by_id[text_id] for text_id in test_ids]

    for text, row in zip(test["text"], rows):
        if text != row["text"]:
            raise ValueError(f"{name}: test texts do not match")

        for start, end in row["spans"]:
            if not 0 <= start < end <= len(text):
                raise ValueError(f"{name}: incorrect test span")

    test_predictions[name] = rows

In [ ]:
test_locations = {}

for name, rows in test_predictions.items():
    locations = []
    for row in rows:
        locations.append(postprocess(row["text"], row["spans"], **CLEAN_OPTIONS))
    test_locations[name] = locations

In [ ]:
SELECTED = None  # "bilstm", "bigru", "union", "intersection", "weighted"
SELECTED_WEIGHTS = {"bilstm": 0.5, "bigru": 0.5}

if SELECTED in test_locations:
    selected = test_locations[SELECTED]
elif SELECTED in ("union", "intersection"):
    selected = combine_predictions(test_locations, SELECTED)
elif SELECTED == "weighted":
    selected = weighted_predictions(test_locations, SELECTED_WEIGHTS)
else:
    raise ValueError("Choose SELECTED by val results")

for text, locations in zip(test["text"], selected):
    for location in locations:
        if location not in text:
            raise ValueError(f"Location is not an original substring: {location!r}")

submission = test[["text_id"]].copy()
submission["locations"] = [json.dumps(locations, ensure_ascii=False) for locations in selected]

output_path = RESULTS_DIR / f"submission_{SELECTED}.csv"
submission.to_csv(output_path, index=False, mode="x")

print(output_path)
display(submission.head())